# Part 2: Streaming application using Spark Structured Streaming  
In this task, you will implement Spark Structured Streaming to consume the data from task 1 and perform a prediction.    
Important: 
- This task uses PySpark Structured Streaming with PySpark Dataframe APIs and PySpark ML.
- You also need your pipeline model from A2A to make predictions and persist the results.
- Note for the prediction related to event time: in a real scenario, you should use accident_ts as the event time; however, since we are simulating streaming and your model was trained with the time column as a feature, you can choose to use the time column or accident_ts.


In [1]:
import os
import shutil

paths_to_delete = [
    "checkpoint",
    "checkpoints",
    "tmp/checkpoint",
    "tmp/checkpoints"
]

for path in paths_to_delete:
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Deleted: {path}")
    else:
        print(f"Not found: {path}")

Deleted: checkpoint
Not found: checkpoints
Not found: tmp/checkpoint
Not found: tmp/checkpoints


1. Write code to create a SparkSession, which 1) uses four cores with a proper application name; 2) uses the UK/London timezone; 3) ensures a checkpoint location has been set.

In [2]:
from pyspark import SparkConf
from pyspark.sql import SparkSession

# Creating Sparksession with the required configurations
conf = SparkConf() \
    .setAppName("A2B_Task2_Spark_Streaming") \
    .setMaster("local[4]") \
    .set("spark.sql.session.timeZone", "Europe/London") \
    .set("spark.sql.streaming.checkpointLocation", "checkpoint/task2_main") \
    .set("spark.sql.shuffle.partitions", "4") \
    .set(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1"
    )

spark = SparkSession.builder.config(conf=conf).getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

print("SparkSession created successfully.")
print("Spark version:", spark.version)

SparkSession created successfully.
Spark version: 4.1.1


2. Write code to define the data schema for the data files. Load the static datasets into data frames. (You can reuse your code from 2A.) In a car accident, we collection streaming information like the realtime road condition, but the vehicle information is static and can be read from the vehicle registration database.

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, LongType, ArrayType

# Establish the schema structure for streaming_collision dataset
streaming_collision_schema = StructType([
    StructField("collision_index", StringType(), True),
    StructField("longitude", StringType(), True),
    StructField("latitude", StringType(), True),
    StructField("date", StringType(), True),
    StructField("time", StringType(), True),
    StructField("road_type", StringType(), True),
    StructField("speed_limit", StringType(), True),
    StructField("junction_detail", StringType(), True),
    StructField("junction_control", StringType(), True),
    StructField("pedestrian_crossing", StringType(), True),
    StructField("light_conditions", StringType(), True),
    StructField("weather_conditions", StringType(), True),
    StructField("road_surface_conditions", StringType(), True),
    StructField("carriageway_hazards", StringType(), True),
    StructField("urban_or_rural_area", StringType(), True),
    StructField("area", StringType(), True),
    StructField("accident_ts", LongType(), True)
])

streaming_collision_array_schema = ArrayType(streaming_collision_schema)

# Establish schema structure for vehicle dataset
vehicle_schema = StructType([
    StructField("collision_index", StringType(), True),
    StructField("vehicle_reference", IntegerType(), True),
    StructField("vehicle_type", IntegerType(), True),
    StructField("vehicle_manoeuvre", IntegerType(), True),
    StructField("junction_location", IntegerType(), True),
    StructField("skidding_and_overturning", IntegerType(), True),
    StructField("hit_object_in_carriageway", IntegerType(), True),
    StructField("first_point_of_impact", IntegerType(), True),
    StructField("sex_of_driver", IntegerType(), True),
    StructField("age_of_driver", IntegerType(), True),
    StructField("engine_capacity_cc", IntegerType(), True),
    StructField("propulsion_code", IntegerType(), True),
    StructField("age_of_vehicle", IntegerType(), True)
])

# Load the vehicle dataset
vehicle_df = spark.read.csv("A2B_dataset/A2B/vehicle.csv", header=True, schema=vehicle_schema)

# Print the vehicle schema and dsiplay first five rows
vehicle_df.printSchema()
vehicle_df.show(5, truncate=False)

root
 |-- collision_index: string (nullable = true)
 |-- vehicle_reference: integer (nullable = true)
 |-- vehicle_type: integer (nullable = true)
 |-- vehicle_manoeuvre: integer (nullable = true)
 |-- junction_location: integer (nullable = true)
 |-- skidding_and_overturning: integer (nullable = true)
 |-- hit_object_in_carriageway: integer (nullable = true)
 |-- first_point_of_impact: integer (nullable = true)
 |-- sex_of_driver: integer (nullable = true)
 |-- age_of_driver: integer (nullable = true)
 |-- engine_capacity_cc: integer (nullable = true)
 |-- propulsion_code: integer (nullable = true)
 |-- age_of_vehicle: integer (nullable = true)

+---------------+-----------------+------------+-----------------+-----------------+------------------------+-------------------------+---------------------+-------------+-------------+------------------+---------------+--------------+
|collision_index|vehicle_reference|vehicle_type|vehicle_manoeuvre|junction_location|skidding_and_overturning|

The above section defines the schemas for the two datasets. The collision stream is defined mostly as strings as kafka message are received in string format, except for "accident_ts", which is a numeric variable. The vehicle dataset is loaded as static as it acts as reference dataset for joins and feature engineering stages later.

3. Using the Kafka topic from the producer in Task 1, ingest the streaming data into Spark Streaming, assuming all data comes in the String format. Except for the 'accident_ts' column, you shall receive it as a numeric type. Then, the data frames should be transformed into the appropriate types.

In [4]:
from pyspark.sql import functions as F

# Configuration for Kafka
hostip = "kafka"
kafka_input_topic = "accident_stream"

# Read the streaming data from the kafka topic from producer
raw_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("subscribe", kafka_input_topic)
    .option("startingOffsets", "latest")
    .load()
)

print("Raw Kafka Dataset schema:")
raw_df.printSchema()

# Convert the kafka value column to string using the array schema
parsed_df = raw_df.select(
    F.from_json(
        F.col("value").cast("string"),
        streaming_collision_array_schema
    ).alias("records")
)

# Explode the JSON array so each accident record becomes one row.
collision_df_exploded = parsed_df.select(F.explode(F.col("records")).alias("record"))

# Convert the nested record structure into normal dataframe columns
streaming_collision_df = collision_df_exploded.select("record.*")

# Display the parsed streaming collision dataset
print("Exploded streaming collision dataset schema:")
streaming_collision_df.printSchema()

# Convert the attributes into their appropriate data types
# try_cast is used for numeric columns so malformed values become NULL instead of terminating the streaming query.

final_collision_df = streaming_collision_df.select(
    F.col("collision_index").cast("string").alias("collision_index"),

    F.expr("try_cast(longitude as double)").alias("longitude"),
    F.expr("try_cast(latitude as double)").alias("latitude"),

    F.col("date").cast("string").alias("date"),
    F.col("time").cast("string").alias("time"),

    F.expr("try_cast(road_type as int)").alias("road_type"),
    F.expr("try_cast(speed_limit as int)").alias("speed_limit"),
    F.expr("try_cast(junction_detail as int)").alias("junction_detail"),
    F.expr("try_cast(junction_control as int)").alias("junction_control"),
    F.expr("try_cast(pedestrian_crossing as int)").alias("pedestrian_crossing"),
    F.expr("try_cast(light_conditions as int)").alias("light_conditions"),
    F.expr("try_cast(weather_conditions as int)").alias("weather_conditions"),
    F.expr("try_cast(road_surface_conditions as int)").alias("road_surface_conditions"),
    F.expr("try_cast(carriageway_hazards as int)").alias("carriageway_hazards"),
    F.expr("try_cast(urban_or_rural_area as int)").alias("urban_or_rural_area"),

    F.col("area").cast("string").alias("area"),
    F.expr("try_cast(accident_ts as long)").alias("accident_ts")
)

print("Final Streaming Collision Dataset Schema:")
final_collision_df.printSchema()

Raw Kafka Dataset schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

Exploded streaming collision dataset schema:
root
 |-- collision_index: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: string (nullable = true)
 |-- speed_limit: string (nullable = true)
 |-- junction_detail: string (nullable = true)
 |-- junction_control: string (nullable = true)
 |-- pedestrian_crossing: string (nullable = true)
 |-- light_conditions: string (nullable = true)
 |-- weather_conditions: string (nullable = true)
 |-- road_surface_conditions: string (nullable = true)
 |-- carriageway_hazards: string (nullable = true)
 |-- ur

This proves that Spark has indeed consumed the data from the Kafka stream. It can be observed from the raw Kafka schema that the Kafka message key and value are obtained as the binary types. The "value" column is first cast as a string, followed by parsing as JSON array before exploding the array into individual accidents. From the intermediate schema, it is clear that all fields from the streaming data are first obtained as strings, apart from "accident_ts," which is a numerical field.
The final schema shows that all the attributes are cast into their appropriate data types.

4. Use a watermark on accident_ts. If data points are received 30 seconds late, discard the data. (note: in a local environment like your laptop, late arrival or delayed processing may never happen.)

In [5]:
# Convert the numeric "accident_ts" attribute into a timestamp column and apply watermark for 30 seconds

collision_df_watermarked = (
    final_collision_df
    .withColumn(
        "event_time",
        F.to_timestamp(F.from_unixtime(F.col("accident_ts")))
    )
    .withWatermark("event_time", "30 seconds")
)

print("Watermarked collision stream schema:")
collision_df_watermarked.printSchema()

Watermarked collision stream schema:
root
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: integer (nullable = true)
 |-- speed_limit: integer (nullable = true)
 |-- junction_detail: integer (nullable = true)
 |-- junction_control: integer (nullable = true)
 |-- pedestrian_crossing: integer (nullable = true)
 |-- light_conditions: integer (nullable = true)
 |-- weather_conditions: integer (nullable = true)
 |-- road_surface_conditions: integer (nullable = true)
 |-- carriageway_hazards: integer (nullable = true)
 |-- urban_or_rural_area: integer (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- event_time: timestamp (nullable = true)



The output clearly confirms that "accident_ts" attribute has been converted into a timestamp column called "event_time". Additionally, a 30 second watermark period is applied so that Spark discards delayed records during later windowed aggregations.

5. Perform the necessary transformation you used in A2A. (note: every student may have used different features, feel free to reuse the code you have written in A2A. If you built an end-to-end pipeline, you can ignore this task.) 

In [6]:
# Filter invalid values in vehicle dataset before aggregation
vehicle_clean_df = (
    vehicle_df
    .withColumn(
        "age_of_driver_clean",
        F.when(
            F.col("age_of_driver").between(16, 100),
            F.col("age_of_driver")
        ).otherwise(None)
    )
    .withColumn(
        "age_of_vehicle_clean",
        F.when(
            F.col("age_of_vehicle").between(0, 80),
            F.col("age_of_vehicle")
        ).otherwise(None)
    )
)

# Aggregate vehicle data by collision_index
vehicle_agg = vehicle_clean_df.groupBy("collision_index").agg(
    F.countDistinct("vehicle_reference").alias("vehicle_count"),

    F.avg("age_of_driver_clean").alias("avg_age_of_driver"),
    F.min("age_of_driver_clean").alias("min_age_of_driver"),
    F.max("age_of_driver_clean").alias("max_age_of_driver"),
    F.avg("age_of_vehicle_clean").alias("avg_age_of_vehicle"),

    # Valid skidding/overturning events only: 1 to 5
    F.sum(
        F.when(F.col("skidding_and_overturning").isin(1, 2, 3, 4, 5), 1)
         .otherwise(0)
    ).alias("skidding_vehicle_count"),

    # Driver sex counts
    F.sum(F.when(F.col("sex_of_driver") == 1, 1).otherwise(0)).alias("male_driver_count"),
    F.sum(F.when(F.col("sex_of_driver") == 2, 1).otherwise(0)).alias("female_driver_count"),
    F.sum(F.when(~F.col("sex_of_driver").isin(1, 2), 1).otherwise(0)).alias("unknown_driver_sex_count"),

    # Young driver count
    F.sum(
        F.when(
            (F.col("age_of_driver_clean").isNotNull()) &
            (F.col("age_of_driver_clean").between(17, 25)),
            1
        ).otherwise(0)
    ).alias("young_driver_count"),

    # Older driver count
    F.sum(
        F.when(
            (F.col("age_of_driver_clean").isNotNull()) &
            (F.col("age_of_driver_clean") >= 65),
            1
        ).otherwise(0)
    ).alias("older_driver_count"),

    # Vehicle type indicators
    F.sum(F.when(F.col("vehicle_type").isin(2, 3, 4, 5, 23, 97, 103, 104, 105, 106), 1).otherwise(0)).alias("motorcycle_count"),
    F.sum(F.when(F.col("vehicle_type").isin(19, 20, 21, 98, 113), 1).otherwise(0)).alias("goods_vehicle_count"),
    F.sum(F.when(F.col("vehicle_type").isin(8, 9, 108, 109), 1).otherwise(0)).alias("car_count"),
    F.sum(F.when(F.col("vehicle_type").isin(10, 11, 110), 1).otherwise(0)).alias("bus_minibus_count"),

    # Manoeuvre-related indicators
    F.sum(F.when(F.col("vehicle_manoeuvre").isin(6, 7, 8, 9, 10), 1).otherwise(0)).alias("turning_vehicle_count"),
    F.sum(F.when(F.col("vehicle_manoeuvre").isin(13, 14, 15), 1).otherwise(0)).alias("overtaking_vehicle_count"),

    # Junction-related vehicle position
    F.sum(F.when(F.col("junction_location").isin(1, 2, 3, 4, 5, 6, 7, 8), 1).otherwise(0)).alias("vehicle_at_junction_count")
)

# Join the watermarked streaming collision data with the aggregated vehicle dataset
joined_collision_vehicle_stream_df = collision_df_watermarked.join(
    vehicle_agg,
    on="collision_index",
    how="left"
)

# Perform feature engineering methods
transformed_stream_df = (
    joined_collision_vehicle_stream_df
    .withColumn(
        "Hour",
        F.hour(F.to_timestamp(F.col("time"), "HH:mm"))
    )
    .withColumn(
        "Peak_Traffic",
        F.when(
            F.col("Hour").isNull(),
            "Unknown_Time"
        ).when(
            ((F.col("Hour") >= 7) & (F.col("Hour") <= 9)) |
            ((F.col("Hour") >= 16) & (F.col("Hour") <= 18)),
            "Peak_Traffic_Hours"
        ).otherwise("Non_Peak_Traffic_Hours")
    )
    .withColumn(
        "Visibility_Weather_Risk",
        F.when(
            F.col("light_conditions").isNull() |
            F.col("weather_conditions").isNull() |
            F.col("light_conditions").isin(-1, 9) |
            F.col("weather_conditions").isin(-1, 9),
            "Unknown_Visibility_or_Weather"
        ).when(
            (F.col("light_conditions").isin(4, 5, 6, 7)) &
            (F.col("weather_conditions").isin(2, 3, 5, 6)),
            "Poor_Visibility_Adverse_Weather"
        ).otherwise("Normal_Visibility_or_Weather")
    )
    .withColumn(
        "location_missing",
        F.when(
            F.col("longitude").isNull() | F.col("latitude").isNull(),
            1
        ).otherwise(0)
    )
    .withColumn(
        "driver_age_missing",
        F.when(F.col("avg_age_of_driver").isNull(), 1).otherwise(0)
    )
    .withColumn(
        "vehicle_age_missing",
        F.when(F.col("avg_age_of_vehicle").isNull(), 1).otherwise(0)
    )
)


# Fill the missing numeric features before passing it to ML pipeline
numeric_fill_values = {
    "vehicle_count": 0,
    "avg_age_of_driver": 0,
    "min_age_of_driver": 0,
    "max_age_of_driver": 0,
    "avg_age_of_vehicle": 0,
    "skidding_vehicle_count": 0,
    "male_driver_count": 0,
    "female_driver_count": 0,
    "unknown_driver_sex_count": 0,
    "young_driver_count": 0,
    "older_driver_count": 0,
    "motorcycle_count": 0,
    "goods_vehicle_count": 0,
    "car_count": 0,
    "bus_minibus_count": 0,
    "turning_vehicle_count": 0,
    "overtaking_vehicle_count": 0,
    "vehicle_at_junction_count": 0,
    "Hour": 0
}

transformed_stream_df = transformed_stream_df.fillna(numeric_fill_values)

# Check the final transformed streaming data schema.
print("Transformed streaming dataset schema:")
transformed_stream_df.printSchema()

Transformed streaming dataset schema:
root
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: integer (nullable = true)
 |-- speed_limit: integer (nullable = true)
 |-- junction_detail: integer (nullable = true)
 |-- junction_control: integer (nullable = true)
 |-- pedestrian_crossing: integer (nullable = true)
 |-- light_conditions: integer (nullable = true)
 |-- weather_conditions: integer (nullable = true)
 |-- road_surface_conditions: integer (nullable = true)
 |-- carriageway_hazards: integer (nullable = true)
 |-- urban_or_rural_area: integer (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- vehicle_count: long (nullable = false)
 |-- avg_age_of_driver: double (nullable = false)
 |-- min_age_of_driver: integer (nullable = fa

6. Load your pipeline model and perform the following aggregations:  
a) Make predictions and print high-severity accidents (>7) every 5 seconds.   
b) Every 10 seconds, print the total number of accidents for each severity.  
c) Every 30 seconds, for each local district with data, print the total number of low (1-3), medium (4-6) and high severity (7-10) accidents.  

In [7]:
# Question-6a: Predict and print high-severity accidents every 5 seconds

from pyspark.ml import PipelineModel

# Load the saved model from Assignment 2A
saved_regression_model_path = "../Assignment_2A/A2A/final_best_severity_model"
severity_pred_model = PipelineModel.load(saved_regression_model_path)

print("Saved A2A model loaded successfully.")


# Use the best model achieved to make predictions on the transfromed streaming data
accident_pred_df = severity_pred_model.transform(transformed_stream_df)


# Round the model prediction to make it a whole number
accident_pred_df = accident_pred_df.withColumn("predicted_severity",F.round(F.col("prediction")).cast("integer"))

# Make sure predicted severity lies within 1 to 10
accident_pred_df = accident_pred_df.withColumn(
    "predicted_severity",
    F.when(F.col("predicted_severity") < 1, 1)
    .when(F.col("predicted_severity") > 10, 10)
    .otherwise(F.col("predicted_severity"))
)


# Assign severity category band levels
accident_pred_df = accident_pred_df.withColumn(
    "severity_level",
    F.when(F.col("predicted_severity").between(1, 3), "Low")
    .when(F.col("predicted_severity").between(4, 6), "Medium")
    .otherwise("High")
)

# Select the accident data with high severity (>7)
high_severity_accidents_df = (
    accident_pred_df
    .filter(F.col("predicted_severity") > 7)
    .select(
        "collision_index",
        "event_time",
        "date",
        "time",
        "area",
        "longitude",
        "latitude",
        "predicted_severity",
        "severity_level"
    )
)

Saved A2A model loaded successfully.


In [8]:
# Print the high severity accident data every 5 seconds
high_severity_query = (
    high_severity_accidents_df
    .writeStream
    .outputMode("append")
    .format("console")
    .option("truncate", False)
    .option("numRows", 20)
    .trigger(processingTime="5 seconds")
    .option("checkpointLocation", "checkpoint/q6a_high_severity_console")
    .start()
)

In [9]:
accident_pred_df.printSchema()

root
 |-- collision_index: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- date: string (nullable = true)
 |-- time: string (nullable = true)
 |-- road_type: integer (nullable = true)
 |-- speed_limit: integer (nullable = true)
 |-- junction_detail: integer (nullable = true)
 |-- junction_control: integer (nullable = true)
 |-- pedestrian_crossing: integer (nullable = true)
 |-- light_conditions: integer (nullable = true)
 |-- weather_conditions: integer (nullable = true)
 |-- road_surface_conditions: integer (nullable = true)
 |-- carriageway_hazards: integer (nullable = true)
 |-- urban_or_rural_area: integer (nullable = true)
 |-- area: string (nullable = true)
 |-- accident_ts: long (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- vehicle_count: long (nullable = false)
 |-- avg_age_of_driver: double (nullable = false)
 |-- min_age_of_driver: integer (nullable = false)
 |-- max_age_of_driver: integer (

In [10]:
# Question 6b: Count the total number of accidents for each predicted severity for every 10 second window

# Group predictions into 10-second windows and count the total number of accidents for each severity level
severity_group_count_df = (
    accident_pred_df
    .groupBy(
        F.window(F.col("event_time"), "10 seconds"),
        F.col("predicted_severity")
    )
    .agg(
        F.count("*").alias("total_accidents")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "predicted_severity",
        "total_accidents"
    )
)

In [11]:
# Print the severity group counts for every 10 seconds
# ALso, use update mode so that the aggregated counts gets printed while windows are active
severity_group_count_query = (
    severity_group_count_df
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .option("numRows", 50)
    .trigger(processingTime="10 seconds")
    .option("checkpointLocation", "checkpoint/q6b_severity_count_console")
    .start()
)

In [12]:
# Question 6c: For every 30 seconds, count the nummber of low, medium, and high severity accidents grouped by area

# Group the accidents into 30-second time window and count low, medium, and high severity accidents for each area
area_severity_count_df = (
    accident_pred_df
    .groupBy(
        F.window(F.col("event_time"), "30 seconds"),
        F.col("area")
    )
    .agg(
        F.sum(
            F.when(F.col("predicted_severity").between(1, 3), 1).otherwise(0)
        ).alias("low_severity_count"),

        F.sum(
            F.when(F.col("predicted_severity").between(4, 6), 1).otherwise(0)
        ).alias("medium_severity_count"),

        F.sum(
            F.when(F.col("predicted_severity").between(7, 10), 1).otherwise(0)
        ).alias("high_severity_count"),

        F.count("*").alias("total_accidents"),

        # Average location for each district/window, used for the bubble map
        F.avg("latitude").alias("district_latitude"),
        F.avg("longitude").alias("district_longitude")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        F.col("area").alias("local_district"),
        "low_severity_count",
        "medium_severity_count",
        "high_severity_count",
        "total_accidents",
        "district_latitude",
        "district_longitude"
    )
)

In [13]:
# Print the area severity group counts for every 30 seconds
# ALso, use update mode so that the aggregated counts gets printed while windows are active
area_severity_count_query = (
    area_severity_count_df
    .writeStream
    .outputMode("update")
    .format("console")
    .option("truncate", False)
    .option("numRows", 50)
    .trigger(processingTime="30 seconds")
    .option("checkpointLocation", "checkpoint/q6c_area_severity_count_console")
    .start()
)

7. Save the data from 6 to Parquet files as streams. (Hint: Parquet files support streaming writing/reading. The file should keep updating while new batches arrive.)

In [14]:
# 7a(save 6a): Save high-severity accident predictions to Parquet file

# Define the parquet path for high severity accident data
high_severity_parquet_path = "parquet/q7a_high_severity_accidents"

# Define checkpoint folder to help Spark track streaming progress
high_severity_parquet_checkpoint = "checkpoint/q7a_high_severity_accidents"


# Write high severity accident data to Parquet from Question 6a as a stream
high_severity_parquet_query = (
    high_severity_accidents_df
    .writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", high_severity_parquet_path)
    .option("checkpointLocation", high_severity_parquet_checkpoint)
    .trigger(processingTime="5 seconds")
    .start()
)

In [15]:
high_severity_parquet_query.status

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [16]:
high_severity_parquet_query.lastProgress

In [17]:
# 7b(save 6b): Save severity grouped count aggregation to Parquet

# Define the parquet path for aggregated severity count
severity_count_parquet_path = "parquet/q7b_severity_count"

# Define checkpoint folder to help Spark track streaming progress
severity_count_parquet_checkpoint = "checkpoint/q7b_severity_count"

# Write the severity count aggregated data to Parquet from Question 6b as a stream
severity_count_parquet_query = (
    severity_group_count_df
    .writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", severity_count_parquet_path)
    .option("checkpointLocation", severity_count_parquet_checkpoint)
    .trigger(processingTime="10 seconds")
    .start()
)

In [18]:
# 7c(save 6c): Save area severity grouped count aggregation to Parquet

# Define the parquet path for area based aggregated severity count
area_severity_parquet_path = "parquet/q7c_area_severity_count"

# Define checkpoint folder to help Spark track streaming progress
area_severity_parquet_checkpoint = "checkpoint/q7c_area_severity_count"

# Write the area based severity count aggregated data to Parquet from Question 6c as a stream
area_severity_parquet_query = (
    area_severity_count_df
    .writeStream
    .outputMode("append")
    .format("parquet")
    .option("path", area_severity_parquet_path)
    .option("checkpointLocation", area_severity_parquet_checkpoint)
    .trigger(processingTime="30 seconds")
    .start()
)

In [19]:
area_severity_parquet_query.status

{'message': 'Initializing sources',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [20]:
area_severity_parquet_query.lastProgress

8. Read the Parquet files from task 7 as data streams and send them to Kafka topics with appropriate names.  
(Note: You shall read the parquet files as a streaming data frame and send messages to the Kafka topic when new data appears in the parquet file.)

In [21]:
# Stream 1

# Read high severity accident Parquet files as a stream and send the records to Kafka topic

# Define the source parquet folder for high severity accidents
q8_stream1_parquet_path = "parquet/q7a_high_severity_accidents"

# Define the Kafka output topic for high severity accidents
q8_stream1_kafka_topic = "q8_high_severity_accidents"

# Define checkpoint folder for this streaming query
q8_stream1_checkpoint = "checkpoint/q8_stream1_high_severity_to_kafka"

# Do static read to get the schema from the existing parquet files
q8_stream1_schema = spark.read.format("parquet").load(q8_stream1_parquet_path).schema

# Read the high severity parquet files as a streaming dataFrame
q8_stream1_df = spark.readStream.schema(q8_stream1_schema).format("parquet").load(q8_stream1_parquet_path)

# Convert the data in each row to JSON string format before sending it to Kafka
q8_stream1_kafka_df = q8_stream1_df.select(
    F.to_json(
        F.struct(
            "collision_index",
            "event_time",
            "date",
            "time",
            "area",
            "longitude",
            "latitude",
            "predicted_severity",
            "severity_level"
        )
    ).alias("value")
)


# Send the JSON records to the Kafka topic
q8_stream1_query = (
    q8_stream1_kafka_df
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", q8_stream1_kafka_topic)
    .option("checkpointLocation", q8_stream1_checkpoint)
    .start()
)

AnalysisException: Unable to infer schema for Parquet at . It must be specified manually.

In [ ]:
q8_stream1_query.status

In [ ]:
q8_stream1_query.lastProgress

In [ ]:
# Stream 2

# Read severity grouped count Parquet files as a stream and send the records to Kafka topic

# Define the source parquet folder for severity grouped count data 
q8_stream2_parquet_path = "parquet/q7b_severity_count"

# Define the Kafka output topic for severity count results
q8_stream2_kafka_topic = "q8_severity_count"

# Define checkpoint folder for this streaming query
q8_stream2_checkpoint = "checkpoint/q8_stream2_severity_count_to_kafka"

# Do static read to get the schema from the existing parquet files
q8_stream2_schema = spark.read.format("parquet").load(q8_stream2_parquet_path).schema

# Read the severity count parquet files as a streaming dataFrame
q8_stream2_df = spark.readStream.schema(q8_stream2_schema).format("parquet").load(q8_stream2_parquet_path)

# Convert each row to JSON string format before sending it to Kafka
q8_stream2_kafka_df = q8_stream2_df.select(
    F.to_json(
        F.struct(
            "window_start",
            "window_end",
            "predicted_severity",
            "total_accidents"
        )
    ).alias("value")
)


# Send the JSON records to the Kafka topic
q8_stream2_query = (
    q8_stream2_kafka_df
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", q8_stream2_kafka_topic)
    .option("checkpointLocation", q8_stream2_checkpoint)
    .start()
)

In [ ]:
q8_stream2_query.status

In [ ]:
q8_stream2_query.lastProgress

In [ ]:
# Stream 3

# Read area wise grouped severity count Parquet files as a stream and send the records to Kafka topic

# Define the source parquet folder for area severity grouped count data
q8_stream3_parquet_path = "parquet/q7c_area_severity_count"

# Define the Kafka output topic for area severity count results
q8_stream3_kafka_topic = "q8_area_severity_count"

# Define checkpoint folder for this streaming query
q8_stream3_checkpoint = "checkpoint/q8_stream3_area_severity_count_to_kafka"

# Do static read to get the schema from the existing parquet files
q8_stream3_schema = spark.read.format("parquet").load(q8_stream3_parquet_path).schema

# Read the area severity count parquet files as a streaming dataFrame
q8_stream3_df = spark.readStream.schema(q8_stream3_schema).format("parquet").load(q8_stream3_parquet_path)

# Convert each row to JSON string format before sending it to Kafka
q8_stream3_kafka_df = q8_stream3_df.select(
    F.to_json(
        F.struct(
            "window_start",
            "window_end",
            "local_district",
            "low_severity_count",
            "medium_severity_count",
            "high_severity_count",
            "total_accidents",
            "district_latitude",
            "district_longitude"
        )
    ).alias("value")
)


# Send the JSON records to the Kafka topic
q8_stream3_query = (
    q8_stream3_kafka_df
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", f"{hostip}:9092")
    .option("topic", q8_stream3_kafka_topic)
    .option("checkpointLocation", q8_stream3_checkpoint)
    .start()
)

In [ ]:
q8_stream3_query.status

In [ ]:
q8_stream3_query.lastProgress

In [ ]:
"""
# For re-running the entire process, un-comment these lines and proceed from Q1-Q8 of Task 2, then run Task 3, finally Task 1.

for query in spark.streams.active:
    query.stop()

paths_to_delete = [
    "checkpoint/q6a_high_severity_console"
    "checkpoint/q7a_high_severity_accidents",
    "parquet/q7a_high_severity_accidents",
    "checkpoint/q8_stream1_high_severity_to_kafka",

    "checkpoint/q6b_severity_count_console"
    "checkpoint/q7b_severity_count",
    "parquet/q7b_severity_count",
    "checkpoint/q8_stream2_severity_count_to_kafka",

    "checkpoint/q6c_area_severity_count_console"
    "parquet/q7c_area_severity_count",
    "checkpoint/q7c_area_severity_count",
    "checkpoint/q8_stream3_area_severity_count_to_kafka"
]

for path in paths_to_delete:
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"Deleted: {path}")
    else:
        print(f"Not found: {path}")
"""

In [ ]:
spark.streams.active